# Time-frequency differences between load levels

Spectrograms (0.1–100 Hz) of selected channels, compared between load levels with a t-test at every time-frequency bin.
Set `PHASE` to `"encoding"` or `"retrieval"`.

In [ ]:
from itertools import combinations
from pathlib import Path
import numpy as np
from helpers import (load_data, bandpass_filter, level_windows, same_length,
                     log_spectrograms, tf_ttest, frequency_band, plot_tmap)

PHASE = "encoding"               # "encoding" or "retrieval"

DATA_DIR = "data"
PATIENT = "kh9_1"
SR = 1024                        # sampling rate (Hz)
BAND = (0.1, 100)                # Hz
LEVELS = ["Level1", "Level2", "Level3"]
ENCODING_POST = 1400             # samples kept after the last encoding event
RETRIEVAL_LENGTH = 1024          # samples from retrieval onset (1 s)
CHANNELS = [26, 35, 116, 117, 105, 81, 82]
ALPHA = 0.05

# spectrogram settings
NPERSEG = 256                    # window length (samples)
NOVERLAP = int(NPERSEG * 0.95)
NFFT = 1024 * 5
FMAX = 100
SIGMA = (200 / 1000) * SR / 2.355    # Gaussian smoothing (bins, along frequency)

OUT_DIR = Path("results") / PHASE / "spectrogram"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load and filter

In [ ]:
data, events, channel_names = load_data(DATA_DIR, PATIENT)
filtered = bandpass_filter(data, *BAND, SR)

## 2. Windows

Same windows as in `02_power`. Encoding windows differ in length, so all are cut or extended
to the 80th percentile of the lengths.

In [ ]:
windows = level_windows(events, PHASE, LEVELS, ENCODING_POST, RETRIEVAL_LENGTH)
windows, length = same_length(windows, percentile=80)
print(f"trial length: {length} samples ({length / SR:.2f} s)")

## 3. Spectrograms and t-tests

For each channel: smoothed log-power spectrogram per trial, then Welch's t-test per time-frequency bin
for each pair of levels. Outlined areas: p < 0.05 (not corrected for the number of bins).

In [ ]:
for ch in CHANNELS:
    name = channel_names[ch]

    spectra = {}
    for level in LEVELS:
        trials = np.stack([filtered[start:end, ch] for start, end in windows[level]])
        f, t, spectra[level] = log_spectrograms(trials, SR, NPERSEG, NOVERLAP, NFFT, FMAX, SIGMA)

    for a, b in combinations(LEVELS, 2):
        t_values, p = tf_ttest(spectra[a], spectra[b])
        sig = p < ALPHA
        plot_tmap(t, f, t_values, sig, title=f"{name}: {a} vs {b} – {PHASE}",
                  save_path=OUT_DIR / f"{name}_{a}_vs_{b}.png")

        # strongest effect
        fi, ti = np.unravel_index(np.nanargmax(np.abs(t_values)), t_values.shape)
        higher = a if t_values[fi, ti] > 0 else b
        print(f"{name}, {a} vs {b}: {100 * sig.mean():.1f} % significant bins, "
              f"peak t = {t_values[fi, ti]:.2f} at {t[ti]:.2f} s, {f[fi]:.1f} Hz "
              f"({frequency_band(f[fi])}), higher in {higher}")